# 环境配置
- conda create -n teach python=3.10
- conda activate teach

# 15数码难题

**教学说明（本科生）：**

本节介绍人工智能中经典的**状态空间搜索**问题。15数码难题是一个典型的规划问题，我们需要判断一个给定的棋盘状态是否可以通过合法移动达到目标状态。

**核心概念：**
- **状态空间**：所有可能的棋盘布局构成的状态集合
- **状态转移**：通过移动空格（用0表示）产生新的状态
- **可解性判定**：并非所有随机打乱的棋盘都有解，需要通过数学方法判断

**学习目标：**
1. 理解如何用逆序数判断棋盘的可解性
2. 掌握奇偶类的概念及其在状态可达性中的应用
3. 为后续学习广度优先搜索、A*算法等寻路算法打下基础

## 单个棋盘是否有解

**教学说明（本科生）：**

这里我们学习如何判断一个15数码棋盘是否"有解"。所谓有解，是指通过移动空格（用0表示），能否将棋盘恢复到有序状态（1-15顺序排列）。

**核心概念：**
- **逆序数**：在忽略空格的情况下，计算所有数字对中前面数字大于后面数字的个数
- **空格行位置**：从底部数空格所在的行数（从1开始计数）
- **可解条件**：当逆序数 + 空格行号为奇数时，棋盘有解

**为什么这样判断？**
这是基于数学上的奇偶类（parity class）理论：每次合法移动只会改变棋盘的奇偶类，因此同一奇偶类内的状态相互可达，不同奇偶类的状态不可达。

In [1]:
def check_solvable(board):
    """
    判断 4×4 十五数码是否可解
    board: 长度为16的列表，0表示空格
    """
    nums = [x for x in board if x != 0]
    inv_count = 0

    # 计算逆序数
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv_count += 1

    # 空格所在行（从底部开始数）
    zero_index = board.index(0)
    row_from_top = zero_index // 4
    row_from_bottom = 4 - row_from_top

    print("当前棋盘：")
    for i in range(0, 16, 4):
        row = board[i:i+4]
        print(" ".join("□" if x == 0 else str(x) for x in row))

    print("\n逆序数 =", inv_count)
    print("空格从底部数第", row_from_bottom, "行")

    if (inv_count + row_from_bottom) % 2 == 1:
        print("结论：该十五数码有解")
    else:
        print("结论：该十五数码无解")


board = [
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 11, 12,
    13, 15, 14, 0
]

check_solvable(board)

当前棋盘：
1 2 3 4
5 6 7 8
9 10 11 12
13 15 14 □

逆序数 = 1
空格从底部数第 1 行
结论：该十五数码无解


## 两个棋盘是否可达

**教学说明（本科生）：**

本节介绍如何判断两个棋盘状态是否可以通过一系列合法移动相互转换。

**核心概念：**
- **奇偶类**：每个棋盘状态都有一个奇偶类（0表示偶类，1表示奇类）
- **可达性**：两个状态可达当且仅当它们具有相同的奇偶类

**应用意义：**
这个概念在游戏开发（如随机生成可解的拼图）、机器人路径规划等领域有广泛应用。理解奇偶类有助于我们设计高效的搜索算法。

In [2]:
def parity_class(board):
    """
    返回 4×4 十五数码的奇偶类
    0 -> 偶类
    1 -> 奇类
    """
    nums = [x for x in board if x != 0]
    inv_count = 0

    # 计算逆序数
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv_count += 1

    # 空格从底部数第几行
    zero_index = board.index(0)
    row_from_top = zero_index // 4
    row_from_bottom = 4 - row_from_top

    return (inv_count + row_from_bottom) % 2


def reachable(board1, board2):
    c1 = parity_class(board1)
    c2 = parity_class(board2)

    print("board1 奇偶类 =", "偶类" if c1 == 0 else "奇类")
    print("board2 奇偶类 =", "偶类" if c2 == 0 else "奇类")

    if c1 == c2:
        print("结论：两个状态可达")
    else:
        print("结论：两个状态不可达")


board1 = [
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 11, 12,
    13, 14, 15, 0
]

board2 = [
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 11, 12,
    13, 15, 14, 0
]

reachable(board1, board2)

board1 奇偶类 = 奇类
board2 奇偶类 = 偶类
结论：两个状态不可达


# 猴子与香蕉问题

**教学说明（本科生）：**

猴子与香蕉问题是人工智能中经典的**规划问题**（Planning Problem），也称为**工具使用问题**。这个问题展示了智能体如何通过组合多个动作来实现目标。

**问题描述：**
一只猴子站在房间中，香蕉悬挂在屋顶某处，旁边有一个箱子。猴子需要通过移动自己和箱子，最终爬到箱子上摘到香蕉。

**核心概念：**
- **状态表示**：用元组 (W, x, Y, z) 表示猴子的位置、是否在箱子上、箱子位置、是否摘到香蕉
- **动作集合**：goto(移动)、pushbox(推箱子)、climbbox(爬箱子)、grasp(摘香蕉)
- **状态空间搜索**：通过BFS等算法寻找从初始状态到目标状态的动作序列

**学习目标：**
1. 理解状态空间的构建方法
2. 掌握如何定义合法的状态转移
3. 体会搜索算法在规划问题中的应用

In [3]:
from collections import deque

# -----------------------------
# 状态定义：
# (W, x, Y, z)
# W: 猴子水平位置
# x: 猴子是否在箱子上，0/1
# Y: 箱子水平位置
# z: 是否摘到香蕉，0/1
# -----------------------------

# 假设房间有 3 个可用位置
POSITIONS = ["A", "B", "C"]

# 设香蕉挂在位置 B 的正上方
BANANA_POS = "B"


def get_next_states(state):
    """
    给定当前状态，返回所有可达的下一步状态
    每个元素为：(动作名称, 新状态)
    """
    W, x, Y, z = state
    next_states = []

    # 如果已经摘到香蕉，就不再扩展
    if z == 1:
        return next_states

    # 1. goto(U): 猴子走到位置 U
    # 条件：猴子必须在地面上（x=0）
    if x == 0:
        for U in POSITIONS:
            if U != W:
                next_states.append((f"goto({U})", (U, 0, Y, z)))

    # 2. pushbox(V): 猴子把箱子推到位置 V
    # 条件：猴子在地面上，且猴子和箱子在同一位置
    if x == 0 and W == Y:
        for V in POSITIONS:
            if V != Y:
                next_states.append((f"pushbox({V})", (V, 0, V, z)))

    # 3. climbbox: 猴子爬上箱子
    # 条件：猴子在地面上，且猴子和箱子在同一位置
    if x == 0 and W == Y:
        next_states.append(("climbbox", (W, 1, Y, z)))

    # 4. grasp: 猴子摘香蕉
    # 条件：猴子在箱子上，箱子在香蕉正下方，尚未摘到香蕉
    if x == 1 and W == Y == BANANA_POS and z == 0:
        next_states.append(("grasp", (W, 1, Y, 1)))

    return next_states


def bfs(initial_state, goal_test):
    """
    广度优先搜索
    返回：动作路径, 状态路径
    """
    queue = deque()
    queue.append((initial_state, [], [initial_state]))
    visited = set()
    visited.add(initial_state)

    while queue:
        current_state, actions, states_path = queue.popleft()

        if goal_test(current_state):
            return actions, states_path

        for action, next_state in get_next_states(current_state):
            if next_state not in visited:
                visited.add(next_state)
                queue.append((
                    next_state,
                    actions + [action],
                    states_path + [next_state]
                ))

    return None, None


def is_goal(state):
    """目标状态：摘到香蕉"""
    return state[3] == 1


# 初始状态示例：
# 猴子在 A，未在箱子上；箱子在 C；还没摘到香蕉
initial_state = ("A", 0, "C", 0)

actions, states = bfs(initial_state, is_goal)

if actions is None:
    print("无解")
else:
    print("找到解！")
    print("\n动作序列：")
    for i, act in enumerate(actions, 1):
        print(f"{i}. {act}")

    print("\n状态变化：")
    for i, st in enumerate(states):
        print(f"{i}: {st}")

找到解！

动作序列：
1. goto(C)
2. pushbox(B)
3. climbbox
4. grasp

状态变化：
0: ('A', 0, 'C', 0)
1: ('C', 0, 'C', 0)
2: ('B', 0, 'B', 0)
3: ('B', 1, 'B', 0)
4: ('B', 1, 'B', 1)


# 修道士和野人问题

**教学说明（本科生）：**

修道士和野人问题是人工智能中经典的**约束满足问题**（Constraint Satisfaction Problem），也是搜索算法教学中的典型案例。

**问题描述：**
有3个修道士和3个野人要过河，只有一条能容纳2人的船。约束条件是：在任何岸边，如果野人数超过修道士人数，野人就会吃掉修道士。需要找到一个安全的过河方案。

**核心概念：**
- **状态定义**：(m, c, b) 表示左岸的修道士数、野人数，以及船的位置
- **安全约束**：任何岸边要么没有修道士，要么修道士人数不少于野人
- **状态空间搜索**：通过BFS等算法寻找从初始状态(3,3,0)到目标状态(0,0,1)的路径

**学习目标：**
1. 理解如何建模带约束的问题
2. 掌握状态空间的构建和遍历方法
3. 体会算法如何找到最优解（最短路径）

In [ ]:
from collections import deque

def is_safe(m, c):
    """
    判断状态是否安全
    m: 左岸修士人数
    c: 左岸野人人数
    """
    # 左岸人数范围
    if not (0 <= m <= 3 and 0 <= c <= 3):
        return False

    # 右岸人数
    mr = 3 - m
    cr = 3 - c

    # 左岸安全：要么没有修士，要么修士不少于野人
    left_safe = (m == 0 or m >= c)

    # 右岸安全：要么没有修士，要么修士不少于野人
    right_safe = (mr == 0 or mr >= cr)

    return left_safe and right_safe


def get_next_states(state):
    """
    根据当前状态生成所有合法下一状态
    状态定义：(m, c, b)
    m: 左岸修士人数
    c: 左岸野人人数
    b: 船的位置，0 表示左岸，1 表示右岸
    """
    m, c, b = state

    # 船一次可载人的5种方式：(修士数, 野人数)
    moves = [
        (1, 0),
        (2, 0),
        (0, 1),
        (0, 2),
        (1, 1)
    ]

    next_states = []

    for dm, dc in moves:
        if b == 0:
            # 船在左岸：从左岸到右岸
            new_m = m - dm
            new_c = c - dc
            new_b = 1
            action = f"左→右: {dm}修士, {dc}野人"
        else:
            # 船在右岸：从右岸到左岸
            new_m = m + dm
            new_c = c + dc
            new_b = 0
            action = f"右→左: {dm}修士, {dc}野人"

        # 人数合法且状态安全
        if is_safe(new_m, new_c):
            next_states.append(((new_m, new_c, new_b), action))

    return next_states


def bfs():
    start = (3, 3, 0)   # 初始状态：左岸3修士3野人，船在左岸
    goal = (0, 0, 1)    # 目标状态：全部到右岸，船在右岸

    queue = deque()
    queue.append((start, []))
    visited = set()
    visited.add(start)

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path + [state]

        for next_state, action in get_next_states(state):
            if next_state not in visited:
                visited.add(next_state)
                queue.append((next_state, path + [state, action]))

    return None


def print_solution(solution):
    if solution is None:
        print("无解")
        return

    print("找到一条合法解：\n")

    for i, item in enumerate(solution):
        if isinstance(item, tuple):
            m, c, b = item
            left_m = m
            left_c = c
            right_m = 3 - m
            right_c = 3 - c
            boat_side = "左岸" if b == 0 else "右岸"

            print(f"状态: 左岸({left_m}修士, {left_c}野人)  "
                  f"右岸({right_m}修士, {right_c}野人)  船在{boat_side}")
        else:
            print(f"步骤{i//2 + 1}: {item}")


if __name__ == "__main__":
    solution = bfs()
    print_solution(solution)

# 鸟类/动物分类问题

**教学说明（本科生）：**

鸟类/动物分类问题是人工智能中经典的**知识表示与推理问题**（Knowledge Representation and Reasoning）。该问题通过动物类别之间的层次关系，以及不同类别所具有的属性，展示知识如何组织、继承和推理。

**问题描述：**  
已知动物、鸟类、哺乳动物等不同类别之间存在层次关系。例如，金丝雀属于鸟类，鸟类属于动物；企鹅也属于鸟类，但企鹅不会飞。系统需要根据已有类别关系和属性规则，判断某个动物属于什么类别，以及它具有哪些属性。

**核心概念：**
- **类别层次表示**：利用 `is-a` 关系描述类别之间的上下位关系，如 `Canary → Bird → Animal`
- **实例关系表示**：利用 `instance-of` 表示具体个体与类别之间的关系，如 `Tweety → Canary`
- **属性继承**：子类和实例可以继承上位类别的属性，如鸟类具有羽毛，因此金丝雀也具有羽毛
- **异常覆盖**：子类可以重新定义父类的默认属性，如鸟类通常会飞，但企鹅的 `can_fly = False`
- **知识推理**：根据已有类别、实例和属性关系，推导未直接存储的新知识

**学习目标：**
1. 理解类别、实例及属性的基本知识表示方法
2. 掌握 `is-a` 和 `instance-of` 两种典型语义关系
3. 理解属性继承及子类属性覆盖机制
4. 体会知识表示如何支持自动推理
5. 初步理解默认推理和异常知识的处理方式

In [3]:
# Bird / Animal Classification
# 用“类别层次 + 属性继承 + 异常覆盖”表示鸟类和动物知识


# =========================
# 1. 类别层次 Knowledge Hierarchy
# =========================

# child -> parent
is_a = {
    "Bird": "Animal",
    "Mammal": "Animal",

    "Canary": "Bird",
    "Penguin": "Bird",
    "Ostrich": "Bird",
    "Eagle": "Bird",

    "Dog": "Mammal",
    "Cat": "Mammal"
}


# =========================
# 2. 类别属性 Properties
# =========================

properties = {
    "Animal": {
        "can_move": True,
        "needs_food": True
    },

    "Bird": {
        "has_wings": True,
        "has_feathers": True,
        "can_fly": True
    },

    "Canary": {
        "color": "yellow",
        "can_sing": True
    },

    "Eagle": {
        "can_fly": True,
        "is_predator": True
    },

    # 异常：企鹅虽然是鸟，但不会飞
    "Penguin": {
        "can_fly": False,
        "can_swim": True
    },

    # 异常：鸵鸟也不会飞
    "Ostrich": {
        "can_fly": False,
        "can_run_fast": True
    },

    "Mammal": {
        "has_hair": True,
        "warm_blooded": True
    },

    "Dog": {
        "sound": "bark"
    },

    "Cat": {
        "sound": "meow"
    }
}


# =========================
# 3. 个体 Instances
# =========================

# individual -> class
instance_of = {
    "Tweety": "Canary",
    "Pingu": "Penguin",
    "Oscar": "Ostrich",
    "Sky": "Eagle",
    "Buddy": "Dog",
    "Kitty": "Cat"
}


# =========================
# 4. 推理函数
# =========================

def get_parent_class(cls):
    """返回某类别的父类别"""
    return is_a.get(cls)


def get_ancestors(cls):
    """
    返回某类别的所有上位类别
    例如：
    Canary -> Bird -> Animal
    """
    ancestors = []

    current = cls

    while current in is_a:
        current = is_a[current]
        ancestors.append(current)

    return ancestors


def is_subclass(child, parent):
    """
    判断 child 是否是 parent 的子类
    """
    if child == parent:
        return True

    return parent in get_ancestors(child)


def get_class(individual):
    """返回个体所属类别"""
    return instance_of.get(individual)


def is_instance(individual, cls):
    """
    判断一个个体是否属于某个类别

    例如：
    Tweety 是 Canary
    Canary 是 Bird
    Bird 是 Animal

    因此：
    Tweety 是 Animal
    """
    direct_class = get_class(individual)

    if direct_class is None:
        return False

    return is_subclass(direct_class, cls)


# =========================
# 5. 属性继承
# =========================

def get_property(individual, prop):
    """
    查询个体的属性。

    优先级：
    个体所属的具体类别
    -> 父类别
    -> 更高层类别

    因此子类属性可以覆盖父类属性。
    """

    cls = get_class(individual)

    if cls is None:
        return None

    current = cls

    while current is not None:

        if current in properties:
            if prop in properties[current]:
                return properties[current][prop]

        current = get_parent_class(current)

    return None


def get_all_properties(individual):
    """
    获取一个个体继承到的全部属性
    """

    cls = get_class(individual)

    if cls is None:
        return {}

    hierarchy = [cls] + get_ancestors(cls)

    # 从最上层开始继承
    hierarchy.reverse()

    result = {}

    for c in hierarchy:
        if c in properties:
            result.update(properties[c])

    return result


# =========================
# 6. 查询 Query
# =========================

print("Tweety 是鸟吗？")
print(is_instance("Tweety", "Bird"))

print()

print("Tweety 是动物吗？")
print(is_instance("Tweety", "Animal"))

print()

print("Tweety 会飞吗？")
print(get_property("Tweety", "can_fly"))

print()

print("Pingu 是鸟吗？")
print(is_instance("Pingu", "Bird"))

print()

print("Pingu 会飞吗？")
print(get_property("Pingu", "can_fly"))

print()

print("Pingu 会游泳吗？")
print(get_property("Pingu", "can_swim"))

print()

print("Oscar 会飞吗？")
print(get_property("Oscar", "can_fly"))

print()

print("Sky 是动物吗？")
print(is_instance("Sky", "Animal"))

print()

print("Tweety 的全部属性：")
print(get_all_properties("Tweety"))

print()

print("Pingu 的全部属性：")
print(get_all_properties("Pingu"))

Tweety 是鸟吗？
True

Tweety 是动物吗？
True

Tweety 会飞吗？
True

Pingu 是鸟吗？
True

Pingu 会飞吗？
False

Pingu 会游泳吗？
True

Oscar 会飞吗？
False

Sky 是动物吗？
True

Tweety 的全部属性：
{'can_move': True, 'needs_food': True, 'has_wings': True, 'has_feathers': True, 'can_fly': True, 'color': 'yellow', 'can_sing': True}

Pingu 的全部属性：
{'can_move': True, 'needs_food': True, 'has_wings': True, 'has_feathers': True, 'can_fly': False, 'can_swim': True}


# 家族关系问题

**教学说明（本科生）：**

家族关系问题是人工智能中经典的**知识表示与逻辑推理问题**（Knowledge Representation and Reasoning）。该问题通过父子、兄弟姐妹、祖父母等关系，展示如何利用事实和规则表示知识，并由已有知识推导新的关系。

**问题描述：**  
已知若干家庭成员及其性别和父母关系。例如，Tom 是 Bob 的父亲，Alice 是 Bob 的母亲，Bob 是 Jack 的父亲。系统需要根据这些基本事实和预先定义的关系规则，推导出兄弟姐妹、祖父母、叔叔、姑姑以及堂表兄弟姐妹等更复杂的家族关系。

**核心概念：**
- **事实表示**：用 `Male(Tom)`、`Parent(Tom, Bob)` 等形式表示已知人物属性和基本关系
- **谓词关系**：利用 `Father(x,y)`、`Sibling(x,y)`、`Grandparent(x,y)` 等谓词描述人物之间的关系
- **规则表示**：通过逻辑规则定义复杂关系，如 `Parent(x,z) ∧ Parent(z,y) → Grandparent(x,y)`
- **关系推理**：根据已有事实和规则，推导知识库中没有直接存储的新关系
- **知识查询**：通过查询某个人的父母、兄弟姐妹或祖父母，验证知识表示和推理结果

**学习目标：**
1. 理解事实、谓词和规则在知识表示中的作用
2. 掌握二元关系的基本表示方法
3. 理解如何通过规则组合已有事实进行逻辑推理
4. 体会显式知识与推导知识之间的区别
5. 初步掌握基于知识库的关系查询方法

In [ ]:
# Family Relations
# 用“事实 + 规则”表示家族关系

# =========================
# 1. 基本事实 Facts
# =========================

male = {
    "Tom",
    "Bob",
    "Jack",
    "Mike"
}

female = {
    "Alice",
    "Mary",
    "Lucy",
    "Emma"
}

# parent(x, y) 表示 x 是 y 的父母
parent = {
    ("Tom", "Bob"),
    ("Alice", "Bob"),

    ("Tom", "Mary"),
    ("Alice", "Mary"),

    ("Bob", "Jack"),
    ("Lucy", "Jack"),

    ("Bob", "Emma"),
    ("Lucy", "Emma"),

    ("Mary", "Mike")
}


# =========================
# 2. 基本关系 Rules
# =========================

def is_parent(x, y):
    """x 是否为 y 的父母"""
    return (x, y) in parent


def is_father(x, y):
    """Father(x, y) = Parent(x, y) AND Male(x)"""
    return is_parent(x, y) and x in male


def is_mother(x, y):
    """Mother(x, y) = Parent(x, y) AND Female(x)"""
    return is_parent(x, y) and x in female


def is_sibling(x, y):
    """
    x 和 y 是否为兄弟姐妹
    条件：
    1. x != y
    2. 至少有一个共同父母
    """
    if x == y:
        return False

    parents_x = {p for p, child in parent if child == x}
    parents_y = {p for p, child in parent if child == y}

    return len(parents_x & parents_y) > 0


def is_brother(x, y):
    """x 是否为 y 的兄弟"""
    return x in male and is_sibling(x, y)


def is_sister(x, y):
    """x 是否为 y 的姐妹"""
    return x in female and is_sibling(x, y)


def is_grandparent(x, y):
    """
    Grandparent(x, y):
    如果 x 是 z 的父母，
    且 z 是 y 的父母，
    则 x 是 y 的祖父母
    """
    for z in get_people():
        if is_parent(x, z) and is_parent(z, y):
            return True
    return False


def is_grandfather(x, y):
    """x 是否为 y 的祖父"""
    return x in male and is_grandparent(x, y)


def is_grandmother(x, y):
    """x 是否为 y 的祖母"""
    return x in female and is_grandparent(x, y)


# =========================
# 3. 更复杂的关系
# =========================

def is_uncle(x, y):
    """
    x 是否为 y 的叔叔/舅舅

    x 是 y 的某个父母的兄弟
    """
    for p in get_people():
        if is_parent(p, y) and is_brother(x, p):
            return True
    return False


def is_aunt(x, y):
    """
    x 是否为 y 的姑姑/姨妈
    """
    for p in get_people():
        if is_parent(p, y) and is_sister(x, p):
            return True
    return False


def is_cousin(x, y):
    """
    x 和 y 是否为堂/表兄弟姐妹

    x 的父母和 y 的父母是兄弟姐妹
    """
    if x == y:
        return False

    parents_x = [p for p, child in parent if child == x]
    parents_y = [p for p, child in parent if child == y]

    for px in parents_x:
        for py in parents_y:
            if is_sibling(px, py):
                return True

    return False


# =========================
# 4. 辅助函数
# =========================

def get_people():
    """返回知识库中的所有人物"""
    people = set()

    people.update(male)
    people.update(female)

    for p, c in parent:
        people.add(p)
        people.add(c)

    return people


def get_parents(person):
    return [
        p for p in get_people()
        if is_parent(p, person)
    ]


def get_children(person):
    return [
        p for p in get_people()
        if is_parent(person, p)
    ]


def get_siblings(person):
    return [
        p for p in get_people()
        if is_sibling(p, person)
    ]


def get_grandparents(person):
    return [
        p for p in get_people()
        if is_grandparent(p, person)
    ]


# =========================
# 5. 查询 Query
# =========================

print("Bob 的父母：", get_parents("Bob"))

print("Jack 的父亲：",
      [x for x in get_people()
       if is_father(x, "Jack")])

print("Jack 的母亲：",
      [x for x in get_people()
       if is_mother(x, "Jack")])

print("Mary 的兄弟姐妹：",
      get_siblings("Mary"))

print("Jack 的祖父母：",
      get_grandparents("Jack"))

print("Mary 是 Jack 的姑姑/姨妈吗？",
      is_aunt("Mary", "Jack"))

print("Bob 是 Mike 的叔叔/舅舅吗？",
      is_uncle("Bob", "Mike"))

print("Jack 和 Mike 是表/堂兄弟吗？",
      is_cousin("Jack", "Mike"))

Bob 的父母： ['Alice', 'Tom']
Jack 的父亲： ['Bob']
Jack 的母亲： ['Lucy']
Mary 的兄弟姐妹： ['Bob']
Jack 的祖父母： ['Alice', 'Tom']
Mary 是 Jack 的姑姑/姨妈吗？ True
Bob 是 Mike 的叔叔/舅舅吗？ True
Jack 和 Mike 是表/堂兄弟吗？ True
